<a href="https://colab.research.google.com/github/azrulartistik/azrulartistik.github.io/blob/main/Sistem_Prediktif_Engagement_Instagram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Instalasi Library

In [1]:
# Instalasi library yang diperlukan
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow xgboost pillow

# 2. Import Data IG STIKOM CKI

In [5]:
# Install instaloader untuk mengambil data IG
!pip install instaloader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 2.4 MB/s eta 0:00:00


In [53]:
# Import Library
import instaloader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import re
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [54]:
# Ambil Data dari Instagram Kampus STIKOM CKI
# Fungsi untuk mengumpulkan data dari Instagram
def collect_instagram_data(username, max_posts=100):
    L = instaloader.Instaloader()
    print(f"Mengambil data dari profil Instagram: {username}")

    try:
        profile = instaloader.Profile.from_username(L.context, username)
        posts_data = []
        count = 0

        for post in profile.get_posts():
            if count >= max_posts:
                break
            post_data = {
                'post_id': post.shortcode,
                'timestamp': post.date_local,
                'caption': post.caption if post.caption else '',
                'likes': post.likes,
                'comments': post.comments,
                'hashtags': list(post.caption_hashtags) if post.caption else [],
                'mentions': list(post.caption_mentions) if post.caption else [],
                'is_video': post.is_video,
                'video_duration': post.video_duration if post.is_video else 0,
                'media_url': post.url,
                'location': post.location.name if post.location else None,
                'weekday': post.date_local.weekday(),
                'hour': post.date_local.hour,
                'engagement': post.likes + post.comments,
            }
            posts_data.append(post_data)
            count += 1

            if count % 10 == 0:
                print(f"Jumlah post yang sudah diambil: {count}")

        df = pd.DataFrame(posts_data)
        print(f"Berhasil mengumpulkan {len(df)} post dari {username}")
        return df

    except instaloader.exceptions.ProfileNotExistsException:
        print(f"Error: Profil Instagram '{username}' tidak ditemukan.")
        return None
    except Exception as e:
        print(f"Error dalam pengambilan data: {e}")
        return None

# Untuk pengujian, saya akan menggunakan jumlah post yang lebih kecil
# Karena akun publik, jadi saya tidak perlu login pada akun tersebut
# df_instagram = collect_instagram_data('stikomcki.jakarta', max_posts=100)

# 3. Pra-pemprosesan Data Engagement IG Kampus

In [55]:
# Jalankan Fungsi untuk Pemprosesan Data
def preprocess_data(df):
    if df is None or df.empty:
        print("Error: DataFrame is None.")
        return None, None, None, None, None, None

    df['caption_length'] = df['caption'].apply(lambda x: len(x))
    df['num_hashtags'] = df['hashtags'].apply(len)
    df['num_mentions'] = df['mentions'].apply(len)

    features = ['caption_length', 'num_hashtags', 'num_mentions', 'weekday', 'hour', 'video_duration']
    X = df[features]
    y = df['engagement']

    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42
    )

    return X_train, X_test, y_train, y_test, scaler, features

# 4. Buat Model LSTM

In [56]:
# Fungsi Model LSTM
def create_lstm_model(input_shape):
    X_train_lstm = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
    X_test_lstm = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

    model = Sequential([
        LSTM(64, input_shape=(1, input_shape), return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])

    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
    model.summary()

    return model, X_train_lstm, X_test_lstm

# Eksekusi Model nya

# Langkah 1: Ambil data dari Instagram kampus
df_instagram = collect_instagram_data('idnboardingschool', max_posts=100)

# Langkah 2: Preprocessing data nya
X_train, X_test, y_train, y_test, scaler, features = preprocess_data(df_instagram)

# Langkah 3: Training dan Evaluasi Model
if X_train is not None:
    lstm_model, X_train_lstm, X_test_lstm = create_lstm_model(X_train.shape[1])

    lstm_history = lstm_model.fit(
        X_train_lstm, y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        verbose=1
    )

    lstm_loss, lstm_mae = lstm_model.evaluate(X_test_lstm, y_test)
    print(f"\n✅ LSTM Test Loss: {lstm_loss}")
    print(f"✅ LSTM Test MAE: {lstm_mae}")

    y_pred_lstm = lstm_model.predict(X_test_lstm).flatten()
else:
    print("❌ Gagal membangun model karena data kosong atau preprocessing gagal.")

Mengambil data dari profil Instagram: idnboardingschool


JSON Query to api/v1/users/web_profile_info/?username=idnboardingschool: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://i.instagram.com/api/v1/users/web_profile_info/?username=idnboardingschool [retrying; skip with ^C]
JSON Query to api/v1/users/web_profile_info/?username=idnboardingschool: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://i.instagram.com/api/v1/users/web_profile_info/?username=idnboardingschool [retrying; skip with ^C]


Error dalam pengambilan data: JSON Query to api/v1/users/web_profile_info/?username=idnboardingschool: 401 Unauthorized - "fail" status, message "Please wait a few minutes before you try again." when accessing https://i.instagram.com/api/v1/users/web_profile_info/?username=idnboardingschool
Error: DataFrame is None.
❌ Gagal membangun model karena data kosong atau preprocessing gagal.


# 5. Buat Model CNN

In [58]:
def create_cnn_model(input_shape):
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

    # Reshape data untuk CNN (samples, time steps, features)
    X_train_cnn = X_train.values.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test_cnn = X_test.values.reshape(X_test.shape[0], X_test.shape[1], 1)

    # Membuat model CNN
    model = Sequential([
        Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(input_shape, 1)),
        MaxPooling1D(pool_size=2),
        Conv1D(filters=32, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1)  # Output layer untuk regresi
    ])

    # Kompilasi model
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

    # Ringkasan model
    model.summary()

    return model, X_train_cnn, X_test_cnn

# Membuat model CNN
cnn_model, X_train_cnn, X_test_cnn = create_cnn_model(X_train.shape[1])

# Training model CNN
cnn_history = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Evaluasi model CNN
cnn_loss, cnn_mae = cnn_model.evaluate(X_test_cnn, y_test)
print(f"CNN Test Loss: {cnn_loss}")
print(f"CNN Test MAE: {cnn_mae}")

# Prediksi dengan model CNN
y_pred_cnn = cnn_model.predict(X_test_cnn).flatten()

AttributeError: 'NoneType' object has no attribute 'shape'

# 6. Buat Model XG-BOOST

In [59]:
def create_xgboost_model():
    import xgboost as xgb

    # Membuat model XGBoost
    model = xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42
    )

    # Training model
    model.fit(X_train, y_train)

    return model

# Membuat model XGBoost
xgb_model = create_xgboost_model()

# Evaluasi model XGBoost
from sklearn.metrics import mean_squared_error, mean_absolute_error
y_pred_xgb = xgb_model.predict(X_test)
xgb_mse = mean_squared_error(y_test, y_pred_xgb)
xgb_mae = mean_absolute_error(y_test, y_pred_xgb)

print(f"XGBoost Test MSE: {xgb_mse}")
print(f"XGBoost Test MAE: {xgb_mae}")

# Feature importance
plt.figure(figsize=(12, 6))
xgb.plot_importance(xgb_model, max_num_features=15)
plt.title('XGBoost Feature Importance')
plt.show()

/usr/local/lib/python3.11/dist-packages/xgboost/data.py:1138: UserWarning: Unknown data type: <class 'NoneType'>, trying to convert it to csr_matrix
  warnings.warn(


TypeError: Not supported type for data.<class 'NoneType'>

# 7. Bagian Ensemble Model

In [60]:
def create_ensemble_predictions(y_pred_lstm, y_pred_cnn, y_pred_xgb):
    # Mengombinasikan prediksi dari ketiga model dengan bobot yang sama
    y_pred_ensemble = (y_pred_lstm + y_pred_cnn + y_pred_xgb) / 3

    # Evaluasi ensemble model
    ensemble_mse = mean_squared_error(y_test, y_pred_ensemble)
    ensemble_mae = mean_absolute_error(y_test, y_pred_ensemble)

    print(f"Ensemble Test MSE: {ensemble_mse}")
    print(f"Ensemble Test MAE: {ensemble_mae}")

    return y_pred_ensemble

# Membuat ensemble predictions
y_pred_ensemble = create_ensemble_predictions(y_pred_lstm, y_pred_cnn, y_pred_xgb)

NameError: name 'y_pred_lstm' is not defined

# 8. Jalankan Ensemble Model Versi Optimal

In [61]:
def find_optimal_weights():
    from scipy.optimize import minimize

    # Fungsi untuk dioptimalkan
    def objective(weights):
        # Normalisasi bobot
        weights = weights / np.sum(weights)

        # Weighted average dari prediksi
        y_pred = weights[0] * y_pred_lstm + weights[1] * y_pred_cnn + weights[2] * y_pred_xgb

        # Return MSE sebagai metrik yang akan diminimalisasi
        return mean_squared_error(y_test, y_pred)

    # Batasan: jumlah bobot = 1
    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})

    # Batas: semua bobot >= 0
    bounds = [(0, 1), (0, 1), (0, 1)]

    # Bobot awal yang sama
    initial_weights = np.array([1/3, 1/3, 1/3])

    # Optimasi
    result = minimize(objective, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)

    # Bobot optimal
    optimal_weights = result['x']
    print(f"Bobot optimal: LSTM = {optimal_weights[0]:.4f}, CNN = {optimal_weights[1]:.4f}, XGBoost = {optimal_weights[2]:.4f}")

    # Prediksi dengan bobot optimal
    y_pred_optimal = optimal_weights[0] * y_pred_lstm + optimal_weights[1] * y_pred_cnn + optimal_weights[2] * y_pred_xgb

    # Evaluasi model ensemble dengan bobot optimal
    optimal_mse = mean_squared_error(y_test, y_pred_optimal)
    optimal_mae = mean_absolute_error(y_test, y_pred_optimal)

    print(f"Ensemble (Optimal Weights) Test MSE: {optimal_mse}")
    print(f"Ensemble (Optimal Weights) Test MAE: {optimal_mae}")

    return optimal_weights, y_pred_optimal

# Mencari bobot optimal
optimal_weights, y_pred_optimal = find_optimal_weights()

NameError: name 'y_pred_lstm' is not defined

# 9. Jalankan Model Evaluasi & Visual

In [62]:
def evaluate_and_visualize():
    # Membuat DataFrame untuk hasil prediksi
    results = pd.DataFrame({
        'Actual': y_test,
        'LSTM': y_pred_lstm,
        'CNN': y_pred_cnn,
        'XGBoost': y_pred_xgb,
        'Ensemble': y_pred_ensemble,
        'Ensemble (Optimal)': y_pred_optimal
    })

    # Menghitung metrik evaluasi
    from sklearn.metrics import r2_score

    models = ['LSTM', 'CNN', 'XGBoost', 'Ensemble', 'Ensemble (Optimal)']
    metrics = {
        'MSE': [],
        'MAE': [],
        'R²': []
    }

    for model in models:
        metrics['MSE'].append(mean_squared_error(results['Actual'], results[model]))
        metrics['MAE'].append(mean_absolute_error(results['Actual'], results[model]))
        metrics['R²'].append(r2_score(results['Actual'], results[model]))

    # Membuat DataFrame untuk metrik
    metrics_df = pd.DataFrame(metrics, index=models)
    print(metrics_df)

    # 1. Perbandingan Aktual vs Prediksi
    plt.figure(figsize=(12, 6))
    plt.scatter(y_test, y_pred_optimal, alpha=0.5)
    plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--')
    plt.xlabel('Actual Engagement')
    plt.ylabel('Predicted Engagement')
    plt.title('Actual vs Predicted Engagement (Optimal Ensemble)')
    plt.grid(True)
    plt.show()

    # 2. Residual Plot
    plt.figure(figsize=(12, 6))
    residuals = y_test - y_pred_optimal
    plt.scatter(y_pred_optimal, residuals, alpha=0.5)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.xlabel('Predicted Engagement')
    plt.ylabel('Residuals')
    plt.title('Residual Plot (Optimal Ensemble)')
    plt.grid(True)
    plt.show()

    # 3. Performance Comparison
    plt.figure(figsize=(12, 6))
    metrics_df['MSE'].plot(kind='bar', color='skyblue')
    plt.title('MSE Comparison Across Models')
    plt.ylabel('Mean Squared Error')
    plt.xticks(rotation=45)
    plt.grid(axis='y')
    plt.show()

    plt.figure(figsize=(12, 6))
    metrics_df['R²'].plot(kind='bar', color='lightgreen')
    plt.title('R² Comparison Across Models')
    plt.ylabel('R² Score')
    plt.xticks(rotation=45)
    plt.grid(axis='y')
    plt.show()

    # 4. Histogram of Engagement
    plt.figure(figsize=(12, 6))
    plt.hist(df['engagement'], bins=30, alpha=0.7)
    plt.xlabel('Engagement')
    plt.ylabel('Frequency')
    plt.title('Distribution of Engagement')
    plt.grid(True)
    plt.show()

    # 5. Heatmap corelation
    plt.figure(figsize=(14, 10))
    correlation = df[features + ['engagement']].corr()
    sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Correlation Matrix')
    plt.tight_layout()
    plt.show()

    return metrics_df

# Evaluasi dan visualisasi
metrics_df = evaluate_and_visualize()

NameError: name 'y_pred_lstm' is not defined

# 10. Model Untuk Prediksi Konten Baru IG Kampus

In [63]:
def predict_new_data():
    # Membuat contoh data baru
    new_data = pd.DataFrame({
        'caption_length': [150, 200, 80],
        'hashtag_count': [5, 2, 8],
        'mention_count': [2, 1, 3],
        'is_video': [1, 0, 1],
        'video_duration': [60, 0, 45],
        'image_brightness': [0.7, 0.5, 0.8],
        'image_contrast': [0.6, 0.7, 0.5],
        'has_emoji': [1, 0, 1],
        'weekday': [5, 3, 6],  # Sabtu, Kamis, Minggu
        'hour': [19, 14, 20],
        'month': [3, 5, 7],
        'category_academic': [0, 1, 0],
        'category_event': [1, 0, 0],
        'category_announcement': [0, 0, 0],
        'category_achievement': [0, 0, 1],
        'category_campus_life': [0, 0, 0]
    })

    # Normalisasi fitur numerik
    numerical_features = [
        'caption_length', 'hashtag_count', 'mention_count', 'video_duration',
        'image_brightness', 'image_contrast', 'weekday', 'hour', 'month'
    ]

    new_data[numerical_features] = scaler.transform(new_data[numerical_features])

    # Prediksi dengan model-model
    # LSTM
    new_data_lstm = new_data.values.reshape(new_data.shape[0], 1, new_data.shape[1])
    pred_lstm = lstm_model.predict(new_data_lstm).flatten()

    # CNN
    new_data_cnn = new_data.values.reshape(new_data.shape[0], new_data.shape[1], 1)
    pred_cnn = cnn_model.predict(new_data_cnn).flatten()

    # XGBoost
    pred_xgb = xgb_model.predict(new_data)

    # Ensemble dengan bobot optimal
    pred_ensemble = (
        optimal_weights[0] * pred_lstm +
        optimal_weights[1] * pred_cnn +
        optimal_weights[2] * pred_xgb
    )

    # Hasil prediksi
    results = pd.DataFrame({
        'LSTM': pred_lstm,
        'CNN': pred_cnn,
        'XGBoost': pred_xgb,
        'Ensemble (Optimal)': pred_ensemble
    })

    print("Prediksi Engagement untuk Data Baru:")
    print(results)

    # Deskripsi data baru
    descriptions = [
        "Post event kampus (video) pada hari Sabtu malam",
        "Post akademik (gambar) pada hari Kamis siang",
        "Post prestasi (video) pada hari Minggu malam"
    ]

    # Membuat visualisasi perbandingan prediksi
    plt.figure(figsize=(12, 8))

    for i in range(len(descriptions)):
        plt.subplot(len(descriptions), 1, i+1)
        plt.bar(['LSTM', 'CNN', 'XGBoost', 'Ensemble'],
                [pred_lstm[i], pred_cnn[i], pred_xgb[i], pred_ensemble[i]],
                color=['blue', 'green', 'orange', 'red'])
        plt.title(f"Prediksi untuk: {descriptions[i]}")
        plt.ylabel('Engagement')
        plt.grid(axis='y')

    plt.tight_layout()
    plt.show()

    return results

# Prediksi untuk data baru
new_data_predictions = predict_new_data()

AttributeError: 'NoneType' object has no attribute 'transform'

# 11. Simpan Model

In [64]:
def save_models():
    # Simpan model LSTM
    lstm_model.save('lstm_model.h5')

    # Simpan model CNN
    cnn_model.save('cnn_model.h5')

    # Simpan model XGBoost
    import pickle
    pickle.dump(xgb_model, open('xgboost_model.pkl', 'wb'))

    # Simpan scaler
    pickle.dump(scaler, open('scaler.pkl', 'wb'))

    # Simpan bobot optimal
    np.save('optimal_weights.npy', optimal_weights)

    print("Model berhasil disimpan.")

# Simpan model
save_models()

NameError: name 'lstm_model' is not defined